In [ ]:
%run "../libs/notebook_init"

In [ ]:
# =============================================================================
# Step log init (STATUS_RUNNING)
# =============================================================================

nb = Utils.get_notebook_context(dbutils)
notebook_folder = nb["notebook_folder"]
notebook_name   = nb["notebook_name"]

step_log_id       = str(uuid.uuid4())
pipeline_run_id   = PIPELINE_RUN_ID
step_sequence     = 1
layer             = "all"
target_table      = None
status            = STATUS_RUNNING
started_timestamp = datetime.now(timezone.utc)
rows_read         = 0
rows_written      = 0
error_message     = None

pipeline_step_log_upsert(
    spark, step_log_id, pipeline_run_id, step_sequence,
    notebook_folder, notebook_name, status, started_timestamp,
    layer, target_table
)


In [ ]:
tables = [
    f"{GOLD}.sales_fact",
    f"{SILVER}.sales",
    f"{SILVER}.dim_product",
    f"{SILVER}.dim_territory",
    f"{SILVER}.dim_store",
    f"{SILVER}.dim_region",
    f"{SILVER}.dim_exchange_rate",
    f"{SILVER}.dim_date",
    f"{SILVER}.dim_currency",
    f"{BRONZE}.products",
    f"{BRONZE}.sales_verde",
    f"{BRONZE}.sales_celeste",
    f"{BRONZE}.sales_arancione",
    f"{AUDIT}.ingestion_log",
    f"{AUDIT}.transform_detail_log",
    f"{AUDIT}.pipeline_step_log",
    f"{AUDIT}.pipeline_log",
]

try:
    for table in tables:
        spark.sql(f"TRUNCATE TABLE {table}")
        print(f"Truncated {table}")

    ended_timestamp = datetime.now(timezone.utc)
    status = STATUS_SUCCEEDED

    pipeline_step_log_upsert(
        spark, step_log_id, pipeline_run_id, step_sequence,
        notebook_folder, notebook_name, status, started_timestamp,
        layer, target_table, rows_read, rows_written, ended_timestamp, error_message
    )

except Exception as e:
    err = Utils.capture_exception(e)
    error_message = (
        f"{err['error_type']}: {err['error_message']}\n\n"
        f"{err['error_traceback']}"
    )

    ended_timestamp = datetime.now(timezone.utc)
    status = STATUS_FAILED
    pipeline_step_log_upsert(
        spark, step_log_id, pipeline_run_id, step_sequence,
        notebook_folder, notebook_name, status, started_timestamp,
        layer, target_table, rows_read, 0, ended_timestamp, error_message
    )
    raise
